In [1]:
pip install sqlalchemy psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ----------------- ---------------------- 1.0/2.4 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 7.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   -------------------------------- ------- 2.4/2.9 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 9.3 MB/s  0:00:00

   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalch

In [ ]:
from sqlalchemy import create_engine


engine = create_engine("sqlite:///test.db", echo=False)


try:
  with engine.connect() as connection:
    print(
        "Zadanie 1: Połączenie z bazą danych test.db zakończyło się"
        " sukcesem za pomocą create_engine!"
    )
except Exception as e:
  print(f"Błąd połączenia: {e}")

Zadanie 1: Połączenie z bazą danych test.db zakończyło się sukcesem za pomocą create_engine!


In [ ]:
#2

from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

engine = create_engine("sqlite:///test.db", echo=False)
Base = declarative_base()



class Student(Base):
  __tablename__ = "students"

  id = Column(Integer, primary_key=True)
  name = Column(String(100))
  age = Column(Integer)
  email = Column(String(150), unique=True)



Student.__table__.create(bind=engine, checkfirst=True)
print("Zadanie 2: Tabela 'students' została utworzona.")


Session = sessionmaker(bind=engine)
session = Session()
print("Struktura tabeli została pomyślnie zainicjalizowana w bazie.")
session.close()


Zadanie 2: Tabela 'students' została utworzona.
Struktura tabeli została pomyślnie zainicjalizowana w bazie.


In [ ]:
#3
from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

engine = create_engine("sqlite:///test.db", echo=False)
Base = declarative_base()


class Student(Base):
  __tablename__ = "students"
  id = Column(Integer, primary_key=True)
  name = Column(String(100))
  age = Column(Integer)
  email = Column(String(150), unique=True)


Student.__table__.create(bind=engine, checkfirst=True)
Session = sessionmaker(bind=engine)
session = Session()


students_data = [
    Student(name="Anna", age=20, email="anna@example.com"),
    Student(name="Piotr", age=22, email="piotr@example.com"),
    Student(name="Katarzyna", age=19, email="katarzyna@example.com"),
    Student(name="Michał", age=24, email="michal@example.com"),
    Student(name="Magda", age=21, email="magda@example.com"),
]

try:
  session.add_all(students_data)
  session.commit()  # Wykonanie commit
  print("Zadanie 3: Dodano 5 studentów i wykonano commit.")
except Exception as e:
  session.rollback()
  print(f"Błąd (być może rekordy już istnieją): {e}")


count = session.query(Student).count()
print(f"Weryfikacja: Liczba rekordów w tabeli wynosi {count}.")
session.close()

Błąd (być może rekordy już istnieją): (sqlite3.IntegrityError) UNIQUE constraint failed: students.email
[SQL: INSERT INTO students (name, age, email) VALUES (?, ?, ?) RETURNING id]
[parameters: ('Anna', 20, 'anna@example.com')]
(Background on this error at: https://sqlalche.me/e/21/gkpj)
Weryfikacja: Liczba rekordów w tabeli wynosi 5.


In [ ]:
#4
from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

engine = create_engine("sqlite:///test.db", echo=False)
Base = declarative_base()


class Student(Base):
  __tablename__ = "students"
  id = Column(Integer, primary_key=True)
  name = Column(String(100))
  age = Column(Integer)
  email = Column(String(150), unique=True)

  def __repr__(self):
    return f"<Student(name='{self.name}', age={self.age})>"


Student.__table__.create(bind=engine, checkfirst=True)
Session = sessionmaker(bind=engine)
session = Session()

print("--- ZADANIE 4: READ ---")


all_students = session.query(Student).all()
print(f"1. Wszyscy studenci: {all_students}")


specific_student = session.query(Student).filter(Student.name == "Anna").first()
print(f"2. Student o imieniu Anna: {specific_student}")


older_than_20 = session.query(Student).filter(Student.age > 20).all()
print(
    "3. Studenci starsi niż 20 lat:",
    [f"{s.name} ({s.age}l)" for s in older_than_20],
)


total_count = session.query(Student).count()
print(f"4. Łączna liczba studentów: {total_count}")

session.close()

--- ZADANIE 4: READ ---
1. Wszyscy studenci: [<Student(name='Anna', age=20)>, <Student(name='Piotr', age=22)>, <Student(name='Katarzyna', age=19)>, <Student(name='Michał', age=24)>, <Student(name='Magda', age=21)>]
2. Student o imieniu Anna: <Student(name='Anna', age=20)>
3. Studenci starsi niż 20 lat: ['Piotr (22l)', 'Michał (24l)', 'Magda (21l)']
4. Łączna liczba studentów: 5


In [ ]:
#5
from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

engine = create_engine("sqlite:///test.db", echo=False)
Base = declarative_base()


class Student(Base):
  __tablename__ = "students"
  id = Column(Integer, primary_key=True)
  name = Column(String(100))
  age = Column(Integer)
  email = Column(String(150), unique=True)


Student.__table__.create(bind=engine, checkfirst=True)
Session = sessionmaker(bind=engine)
session = Session()

print("--- ZADANIE 5: UPDATE ---")


student_to_update = session.query(Student).filter(Student.name == "Anna").first()
if student_to_update:
  student_to_update.age = 22
  session.commit()
  print(f"Zaktualizowano wiek Anny na: {student_to_update.age}")


session.query(Student).update({Student.age: Student.age + 1})
session.commit()
print("Wykonano bulk update (zwiększono wiek wszystkich studentów o 1 rok).")


for s in session.query(Student).all():
  print(f"Student: {s.name}, Nowy wiek: {s.age}")

session.close()

--- ZADANIE 5: UPDATE ---
Zaktualizowano wiek Anny na: 22
Wykonano bulk update (zwiększono wiek wszystkich studentów o 1 rok).
Student: Anna, Nowy wiek: 23
Student: Piotr, Nowy wiek: 23
Student: Katarzyna, Nowy wiek: 20
Student: Michał, Nowy wiek: 25
Student: Magda, Nowy wiek: 22


In [ ]:
#6
from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

engine = create_engine("sqlite:///test.db", echo=False)
Base = declarative_base()


class Student(Base):
  __tablename__ = "students"
  id = Column(Integer, primary_key=True)
  name = Column(String(100))
  age = Column(Integer)
  email = Column(String(150), unique=True)


Student.__table__.create(bind=engine, checkfirst=True)
Session = sessionmaker(bind=engine)
session = Session()

print("--- ZADANIE 6: DELETE ---")


youngest = session.query(Student).order_by(Student.age.asc()).first()
if youngest:
  print(
      f"Usuwanie najmłodszego studenta: {youngest.name} (wiek: {youngest.age})"
  )
  session.delete(youngest)
  session.commit()


deleted_count = (
    session.query(Student)
    .filter(Student.age > 24)
    .delete(synchronize_session=False)
)
session.commit()
print(f"Wykonano bulk delete. Usunięto rekordów: {deleted_count}")

remaining_count = session.query(Student).count()
print(f"Weryfikacja: Liczba pozostałych studentów w bazie: {remaining_count}")

session.close()

--- ZADANIE 6: DELETE ---
Usuwanie najmłodszego studenta: Katarzyna (wiek: 20)
Wykonano bulk delete. Usunięto rekordów: 1
Weryfikacja: Liczba pozostałych studentów w bazie: 3


In [ ]:
#7
from sqlalchemy import Column, ForeignKey, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

engine = create_engine("sqlite:///test.db", echo=False)
Base = declarative_base()



class Teacher(Base):
  __tablename__ = "teachers"
  id = Column(Integer, primary_key=True)
  name = Column(String(100))
  courses = relationship("Course", back_populates="teacher")


class Course(Base):
  __tablename__ = "courses"
  id = Column(Integer, primary_key=True)
  title = Column(String(100))
  teacher_id = Column(Integer, ForeignKey("teachers.id"))
  teacher = relationship("Teacher", back_populates="courses")



Teacher.__table__.create(bind=engine, checkfirst=True)
Course.__table__.create(bind=engine, checkfirst=True)

Session = sessionmaker(bind=engine)
session = Session()

print("--- ZADANIE 7: Relacja One-to-Many ---")


teacher = Teacher(name="Jan Profesor")
teacher.courses = [
    Course(title="Matematyka Dyskretna"),
    Course(title="Bazy Danych"),
    Course(title="Algorytmy i Struktury Danych"),
]

session.add(teacher)
session.commit()


fetched_teacher = (
    session.query(Teacher).filter(Teacher.name == "Jan Profesor").first()
)
print(f"Nauczyciel: {fetched_teacher.name}")
print("Wypisanie kursów przypisanych do nauczyciela:")
for course in fetched_teacher.courses:
  print(f" - {course.title}")

session.close()

--- ZADANIE 7: Relacja One-to-Many ---
Nauczyciel: Jan Profesor
Wypisanie kursów przypisanych do nauczyciela:
 - Matematyka Dyskretna
 - Bazy Danych
 - Algorytmy i Struktury Danych


In [ ]:
#8
import pandas as pd
from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

engine = create_engine("sqlite:///test.db", echo=False)
Base = declarative_base()


class Product(Base):
  __tablename__ = "products"
  id = Column(Integer, primary_key=True)
  name = Column(String(100))
  category = Column(String(50))
  price = Column(Integer)



Product.__table__.create(bind=engine, checkfirst=True)
Session = sessionmaker(bind=engine)
session = Session()

print("--- ZADANIE 8: Pandas + ORM ---")


data = {
    "name": [
        "Laptop",
        "Myszka",
        "Klawiatura",
        "Biurko",
        "Krzesło",
        "Lampa",
        "Python od podstaw",
        "Czysty Kod",
        "Monitor",
        "Słuchawki",
    ],
    "category": [
        "Electronics",
        "Electronics",
        "Electronics",
        "Furniture",
        "Furniture",
        "Furniture",
        "Books",
        "Books",
        "Electronics",
        "Electronics",
    ],
    "price": [3500, 150, 300, 800, 500, 120, 90, 110, 1200, 250],
}
df = pd.DataFrame(data)


products_list = [
    Product(
        name=row["name"], category=row["category"], price=int(row["price"])
    )
    for _, row in df.iterrows()
]

try:
  session.add_all(products_list)
  session.commit()
  print("Przekonwertowano DataFrame i zapisano 10 produktów do bazy danych.")
except Exception as e:
  session.rollback()
  print(f"Błąd zapisu (mogą już istnieć): {e}")


db_products = session.query(Product).all()
db_df = pd.DataFrame(
    [
        {"name": p.name, "category": p.category, "price": p.price}
        for p in db_products
    ]
)

print("\nOdczytane dane z bazy (pierwsze 5 wierszy):")
print(db_df.head())

session.close()

--- ZADANIE 8: Pandas + ORM ---
Przekonwertowano DataFrame i zapisano 10 produktów do bazy danych.

Odczytane dane z bazy (pierwsze 5 wierszy):
         name     category  price
0      Laptop  Electronics   3500
1      Myszka  Electronics    150
2  Klawiatura  Electronics    300
3      Biurko    Furniture    800
4     Krzesło    Furniture    500
